# Fine-Tune a Generative AI Model for Dialogue Summarization

Here I am just sharing an interesting tutorial that was included as part of a course that I recently audited on Coursera.

I thought that this tutorial did a great job of explaining parameter efficient fine-tuning methods (as well as providing examples for how to implement these methods).


Source:
 - https://www.coursera.org/learn/generative-ai-with-llms
 - https://www.coursera.org/learn/generative-ai-with-llms/gradedLti/x0gc1/lab-2-fine-tune-a-generative-ai-model-for-dialogue-summarization

https://creativecommons.org/licenses/by-sa/2.0/legalcode

In this notebook, you will fine-tune an existing LLM from Hugging Face for enhanced dialogue summarization. You will use the [FLAN-T5](https://huggingface.co/docs/transformers/model_doc/flan-t5) model, which provides a high quality instruction tuned model and can summarize text out of the box. To improve the inferences, you will explore a full fine-tuning approach and evaluate the results with ROUGE metrics. Then you will perform Parameter Efficient Fine-Tuning (PEFT), evaluate the resulting model and see that the benefits of PEFT outweigh the slightly-lower performance metrics.

<a name='1'></a>
## 1 - Set up Kernel, Load Required Dependencies, Dataset and LLM

<a name='1.1'></a>
### 1.1 - Set up Kernel and Required Dependencies

Now install the required packages for the LLM and datasets.



In [ ]:
import sys
import subprocess
import platform

print(f"Python version: {sys.version}")
print(f"System info: {platform.platform()}")

# Check Python version
python_version = sys.version_info
print(f"Python major.minor: {python_version.major}.{python_version.minor}")

# Define packages with version constraints based on Python version
if python_version.minor >= 11:  # Python 3.11+
    packages = [
        "--upgrade pip",
        "torch>=2.0.0",
        "torchdata>=0.7.0",
        "torchvision>=0.15.0",
        "transformers>=4.44.0",
        "datasets>=3.0.0",
        "evaluate",
        "rouge_score",
        "loralib",
        "peft>=0.12.0",
        "tokenizers>=0.19.0",
        "accelerate",
        "sentencepiece",
        "protobuf<4.24.0"  # Compatibility fix for Python 3.11+
    ]
else:  # Python 3.10 and below
    packages = [
        "--upgrade pip",
        "torch>=1.13.0",
        "torchdata>=0.5.0",
        "torchvision>=0.14.0",
        "transformers>=4.44.0",
        "datasets>=3.0.0",
        "evaluate",
        "rouge_score",
        "loralib",
        "peft>=0.12.0",
        "tokenizers>=0.19.0",
        "accelerate",
        "sentencepiece"
    ]

# Install packages
for package in packages:
    if package.startswith("--"):
        result = subprocess.run(["pip", "install", package], capture_output=True, text=True)
    else:
        result = subprocess.run(["pip", "install", package, "--quiet"], capture_output=True, text=True)

    if result.returncode != 0:
        print(f"Warning: Issue installing {package}: {result.stderr}")

print("Package installation completed!")

# Install awscli
subprocess.run(["pip", "install", "awscli", "--quiet"], check=True)
print("awscli installed successfully!")

# Import libraries after installation
try:
    from datasets import load_dataset
    from transformers import (
        AutoModelForSeq2SeqLM,
        AutoTokenizer,
        GenerationConfig,
        TrainingArguments,
        Trainer
    )
    import torch
    import time
    import evaluate
    import pandas as pd
    import numpy as np
    print("✓ All imports successful!")
except Exception as e:
    print(f"Import error: {e}")
    print("Attempting to install missing dependencies...")
    # If imports fail, try installing with more flexible versions
    subprocess.run(["pip", "install", "datasets", "transformers", "torch"], check=True)
    from datasets import load_dataset
    from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
    import torch

# Verify CUDA availability
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


Python version: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
System info: Linux-6.6.113+-x86_64-with-glibc2.35
Python major.minor: 3.12
Usage:   
  pip3 install [options] <requirement specifier> [package-index-options] ...
  pip3 install [options] -r <requirements file> [package-index-options] ...
  pip3 install [options] [-e] <vcs project url> ...
  pip3 install [options] [-e] <local project path> ...
  pip3 install [options] <archive url/path> ...

no such option: --upgrade pip

Package installation completed!
awscli installed successfully!
✓ All imports successful!

PyTorch version: 2.10.0+cu128
CUDA available: True
CUDA device: Tesla T4
CUDA memory: 15.64 GB


<a name='1.2'></a>
### 1.2 - Load Dataset and LLM

You are going to continue experimenting with the [DialogSum](https://huggingface.co/datasets/knkarthick/dialogsum) Hugging Face dataset. It contains 10,000+ dialogues with the corresponding manually labeled summaries and topics.

In [ ]:
huggingface_dataset_name = "knkarthick/dialogsum"

dataset = load_dataset(huggingface_dataset_name)

dataset

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.csv:   0%|          | 0.00/11.3M [00:00<?, ?B/s]

validation.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/12460 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1500 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 12460
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 500
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 1500
    })
})

Load the pre-trained [FLAN-T5 model](https://huggingface.co/docs/transformers/model_doc/flan-t5) and its tokenizer directly from HuggingFace. Notice that you will be using the [small version](https://huggingface.co/google/flan-t5-base) of FLAN-T5. Setting `torch_dtype=torch.bfloat16` specifies the memory type to be used by this model.

In [ ]:
model_name='google/flan-t5-base'
# model_name='/kaggle/input/flan-t5/pytorch/base/4'
original_model = AutoModelForSeq2SeqLM.from_pretrained(model_name, torch_dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

It is possible to pull out the number of model parameters and find out how many of them are trainable. The following function can be used to do that, at this stage, you do not need to go into details of it.

In [ ]:
def print_number_of_trainable_model_parameters(model):
    trainable_model_params = 0
    all_model_params = 0
    for _, param in model.named_parameters():
        all_model_params += param.numel()
        if param.requires_grad:
            trainable_model_params += param.numel()
    return f"trainable model parameters: {trainable_model_params}\nall model parameters: {all_model_params}\npercentage of trainable model parameters: {100 * trainable_model_params / all_model_params:.2f}%"

print(print_number_of_trainable_model_parameters(original_model))

trainable model parameters: 247577856
all model parameters: 247577856
percentage of trainable model parameters: 100.00%


<a name='1.3'></a>
### 1.3 - Test the Model with Zero Shot Inferencing

Test the model with the zero shot inferencing. You can see that the model struggles to summarize the dialogue compared to the baseline summary, but it does pull out some important information from the text which indicates the model can be fine-tuned to the task at hand.

In [ ]:
index = 200

dialogue = dataset['test'][index]['dialogue']
summary = dataset['test'][index]['summary']

prompt = f"""
Summarize the following conversation.

{dialogue}

Summary:
"""

inputs = tokenizer(prompt, return_tensors='pt')
output = tokenizer.decode(
    original_model.generate(
        inputs["input_ids"],
        max_new_tokens=200,
    )[0],
    skip_special_tokens=True
)

dash_line = '-'.join('' for x in range(100))
print(dash_line)
print(f'INPUT PROMPT:\n{prompt}')
print(dash_line)
print(f'BASELINE HUMAN SUMMARY:\n{summary}\n')
print(dash_line)
print(f'MODEL GENERATION - ZERO SHOT:\n{output}')

---------------------------------------------------------------------------------------------------
INPUT PROMPT:

Summarize the following conversation.

#Person1#: Have you considered upgrading your system?
#Person2#: Yes, but I'm not sure what exactly I would need.
#Person1#: You could consider adding a painting program to your software. It would allow you to make up your own flyers and banners for advertising.
#Person2#: That would be a definite bonus.
#Person1#: You might also want to upgrade your hardware because it is pretty outdated now.
#Person2#: How can we do that?
#Person1#: You'd probably need a faster processor, to begin with. And you also need a more powerful hard disc, more memory and a faster modem. Do you have a CD-ROM drive?
#Person2#: No.
#Person1#: Then you might want to add a CD-ROM drive too, because most new software programs are coming out on Cds.
#Person2#: That sounds great. Thanks.

Summary:

-------------------------------------------------------------------

In [ ]:
# Parameter Efficient Fine-Tuning (PEFT)
# Here we configure LoRA (Low-Rank Adaptation) for fine-tuning instead of updating all model parameters.

# Import required PEFT components
# - LoraConfig: Used to define LoRA hyperparameters
# - get_peft_model: Wraps the base model with LoRA adapters
# - TaskType: Specifies the type of NLP task
from peft import LoraConfig, get_peft_model, TaskType


# Define LoRA configuration
lora_config = LoraConfig(

    # r = rank of the low-rank matrices
    # Higher r → more trainable parameters → more expressive but heavier
    r=32,

    # Scaling factor for LoRA updates
    # Controls how much the LoRA updates influence the base model
    lora_alpha=32,

    # Specify which layers to apply LoRA to
    # "q" and "v" correspond to Query and Value projection matrices
    # inside the attention mechanism
    target_modules=["q", "v"],

    # Dropout applied to LoRA layers to reduce overfitting
    lora_dropout=0.05,

    # Whether to train bias parameters
    # "none" means biases are kept frozen
    bias="none",

    # Specify the task type
    # SEQ_2_SEQ_LM = Sequence-to-Sequence Language Modeling
    # Used for models like FLAN-T5 (summarization, translation, etc.)
    task_type=TaskType.SEQ_2_SEQ_LM
)

In [ ]:
#Add LoRA adapter layers/parameters to the original LLM to be trained.
peft_model = get_peft_model(original_model,
                            lora_config)
print(print_number_of_trainable_model_parameters(peft_model))

trainable model parameters: 3538944
all model parameters: 251116800
percentage of trainable model parameters: 1.41%


In [ ]:
def tokenize_function(example):
    # Define the instruction prefix (what we want the model to do)
    start_prompt = 'Summarize the following conversation.\n\n'

    # Define the suffix that signals where the summary should start
    end_prompt = '\n\nSummary: '

    # Create full prompts by combining:
    # instruction + dialogue text + summary keyword
    # example["dialogue"] is a batch (list) because batched=True
    prompt = [
        start_prompt + dialogue + end_prompt
        for dialogue in example["dialogue"]
    ]

    # Tokenize the input prompts:
    # - padding="max_length" → pad all sequences to same length
    # - truncation=True → cut off text if too long
    # - return_tensors="pt" → return PyTorch tensors
    example['input_ids'] = tokenizer(
        prompt,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    ).input_ids

    # Tokenize the target summaries (these become labels for training)
    # During training, the model tries to predict these tokens
    example['labels'] = tokenizer(
        example["summary"],
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    ).input_ids

    # Return the modified batch
    return example


# The dataset contains three splits:
# - train
# - validation
# - test
#
# map() applies tokenize_function to ALL splits.
# batched=True means examples are processed in batches instead of one-by-one.
tokenized_datasets = dataset.map(tokenize_function, batched=True)


# Remove unused columns to keep only what the model needs:
# We no longer need raw text fields after tokenization.
tokenized_datasets = tokenized_datasets.remove_columns([
    'id',
    'topic',
    'dialogue',
    'summary',
])

Map:   0%|          | 0/12460 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

In [ ]:
# Reduce the dataset size by keeping only every 100th example
# This is typically done to:
# - Speed up experimentation
# - Reduce training time
# - Run quick debugging tests

# filter() applies a condition to each example
# with_indices=True allows us to access the index of each example

tokenized_datasets = tokenized_datasets.filter(

    # Keep the example only if its index is divisible by 100
    # (i.e., keep index 0, 100, 200, 300, ...)
    lambda example, index: index % 100 == 0,

    # Enables access to the index inside the lambda function
    with_indices=True
)

Filter:   0%|          | 0/12460 [00:00<?, ? examples/s]

Filter:   0%|          | 0/500 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1500 [00:00<?, ? examples/s]

In [ ]:
print(f"Shapes of the datasets:")
print(f"Training: {tokenized_datasets['train'].shape}")
print(f"Validation: {tokenized_datasets['validation'].shape}")
print(f"Test: {tokenized_datasets['test'].shape}")

print(tokenized_datasets)

Shapes of the datasets:
Training: (125, 2)
Validation: (5, 2)
Test: (15, 2)
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 125
    })
    validation: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 5
    })
    test: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 15
    })
})


In [ ]:
output_dir = f'./peft-dialogue-summary-training-{str(int(time.time()))}'

# Define the fraction of the dataset you want to use (e.g., 10%)
subset_fraction = 0.1

# Calculate the number of samples to use
subset_size = int(len(dataset["train"]) * subset_fraction)

# Create a subset of the dataset
subset_dataset = dataset["train"].shuffle(seed=42).select(range(subset_size))

# Tokenize the subset
tokenized_subset = subset_dataset.map(tokenize_function, batched=True)


print(tokenized_subset)
# Define training arguments and create Trainer instance.

peft_training_args = TrainingArguments(
    output_dir=output_dir,
    auto_find_batch_size=4,
    learning_rate=1e-3, # Higher learning rate than full fine-tuning.
    num_train_epochs=10,
    logging_steps=1,
    max_steps=10
)

peft_trainer = Trainer(
    model=peft_model,
    args=peft_training_args,
    train_dataset=tokenized_subset,
)


Map:   0%|          | 0/1246 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'dialogue', 'summary', 'topic', 'input_ids', 'labels'],
    num_rows: 1246
})


In [ ]:
peft_trainer.train()

peft_model_path="./peft-dialogue-summary-checkpoint-local"

peft_trainer.model.save_pretrained(peft_model_path)
tokenizer.save_pretrained(peft_model_path)

Step,Training Loss
1,11.500000
2,11.375000
3,11.250000
4,11.187500
5,11.062500
6,11.000000
7,10.937500
8,10.937500
9,10.875000
10,10.875000


('./peft-dialogue-summary-checkpoint-local/tokenizer_config.json',
 './peft-dialogue-summary-checkpoint-local/tokenizer.json')

In [ ]:
from peft import PeftModel, PeftConfig

peft_model_base = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base", torch_dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")

peft_model = PeftModel.from_pretrained(peft_model_base,
                                       './peft-dialogue-summary-checkpoint-local',
                                       torch_dtype=torch.bfloat16,
                                       is_trainable=False)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [ ]:
#The number of trainable parameters will be 0 due to is_trainable=False setting:
print(print_number_of_trainable_model_parameters(peft_model))

trainable model parameters: 3538944
all model parameters: 251116800
percentage of trainable model parameters: 1.41%


In [ ]:
index = 200
dialogue = dataset['test'][index]['dialogue']
human_baseline_summary = dataset['test'][index]['summary']

prompt = f"""
Summarize the following conversation.

{dialogue}

Summary:
"""

# Tokenize the input prompt and make sure it's on the same device as the model
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(original_model.device)

original_model_outputs = original_model.generate(input_ids=input_ids, generation_config=GenerationConfig(max_new_tokens=200, num_beams=1))
original_model_text_output = tokenizer.decode(original_model_outputs[0], skip_special_tokens=True)

print(dash_line)
print(f'BASELINE HUMAN SUMMARY:\n{human_baseline_summary}')
print(dash_line)
print(f'ORIGINAL MODEL:\n{original_model_text_output}')


---------------------------------------------------------------------------------------------------
BASELINE HUMAN SUMMARY:
#Person1# teaches #Person2# how to upgrade software and hardware in #Person2#'s system.
---------------------------------------------------------------------------------------------------
ORIGINAL MODEL:
Upgrade your computer to a new processor and a more powerful processor.


In [ ]:
index = 200
human_baseline_summary = dataset['test'][index]['summary']
dialogue = dataset['test'][index]['dialogue']

prompt = f"""
Summarize the following conversation.

{dialogue}

Summary: """

# Tokenize the input prompt and make sure it's on the same device as the models
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(original_model.device)

# Move the PEFT model to the same device as the original model
peft_model = peft_model.to(original_model.device)

# Generate summaries using the original model and PEFT model
with torch.no_grad():
    original_model_outputs = original_model.generate(input_ids=input_ids, generation_config=GenerationConfig(max_new_tokens=200, num_beams=1))
    peft_model_outputs = peft_model.generate(input_ids=input_ids, generation_config=GenerationConfig(max_new_tokens=200, num_beams=1))

original_model_text_output = tokenizer.decode(original_model_outputs[0], skip_special_tokens=True)
peft_model_text_output = tokenizer.decode(peft_model_outputs[0], skip_special_tokens=True)

print(dash_line)
print(f'BASELINE HUMAN SUMMARY:\n{human_baseline_summary}')
print(dash_line)
print(f'ORIGINAL MODEL:\n{original_model_text_output}')
print(dash_line)
print(f'PEFT MODEL: {peft_model_text_output}')


---------------------------------------------------------------------------------------------------
BASELINE HUMAN SUMMARY:
#Person1# teaches #Person2# how to upgrade software and hardware in #Person2#'s system.
---------------------------------------------------------------------------------------------------
ORIGINAL MODEL:
You are welcome.
---------------------------------------------------------------------------------------------------
PEFT MODEL: I'm looking for a new computer.


In [ ]:
rouge = evaluate.load('rouge')


In [ ]:
dialogues = dataset['test'][0:10]['dialogue']
human_baseline_summaries = dataset['test'][0:10]['summary']

original_model_summaries = []
peft_model_summaries = []

# Move both models to the same device (CPU or GPU)
device = original_model.device  # Assuming original_model and peft_model are on the same device
original_model = original_model.to(device)
peft_model = peft_model.to(device)

for idx, dialogue in enumerate(dialogues):
    prompt = f"""
Summarize the following conversation.

{dialogue}

Summary: """

    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

    human_baseline_text_output = human_baseline_summaries[idx]

    with torch.no_grad():
        original_model_outputs = original_model.generate(input_ids=input_ids, generation_config=GenerationConfig(max_new_tokens=200))
        original_model_text_output = tokenizer.decode(original_model_outputs[0], skip_special_tokens=True)

        peft_model_outputs = peft_model.generate(input_ids=input_ids, generation_config=GenerationConfig(max_new_tokens=200))
        peft_model_text_output = tokenizer.decode(peft_model_outputs[0], skip_special_tokens=True)

    original_model_summaries.append(original_model_text_output)
    peft_model_summaries.append(peft_model_text_output)

zipped_summaries = list(zip(human_baseline_summaries, original_model_summaries, peft_model_summaries))

df = pd.DataFrame(zipped_summaries, columns=['human_baseline_summaries', 'original_model_summaries', 'peft_model_summaries'])


In [ ]:
rouge = evaluate.load('rouge')

original_model_results = rouge.compute(
    predictions=original_model_summaries,
    references=human_baseline_summaries[0:len(original_model_summaries)],
    use_aggregator=True,
    use_stemmer=True,
)


peft_model_results = rouge.compute(
    predictions=peft_model_summaries,
    references=human_baseline_summaries[0:len(peft_model_summaries)],
    use_aggregator=True,
    use_stemmer=True,
)

print('ORIGINAL MODEL:')
print(original_model_results)
print('PEFT MODEL:')
print(peft_model_results)

ORIGINAL MODEL:
{'rouge1': np.float64(0.29043219910291584), 'rouge2': np.float64(0.12699801079852233), 'rougeL': np.float64(0.24648227653379823), 'rougeLsum': np.float64(0.24976767958390714)}
PEFT MODEL:
{'rouge1': np.float64(0.2569926860093741), 'rouge2': np.float64(0.12144803666542796), 'rougeL': np.float64(0.2217442105043362), 'rougeLsum': np.float64(0.2246290630251057)}
